# Data Understanding

Notebook ini membaca file mentah dari `data/raw` tanpa memodifikasi isinya. Analisis mencakup profil seluruh file, validasi kalender perdagangan, kualitas volume, konsistensi pertumbuhan PDB, pemeriksaan nilai ekstrem, dan `Data Quality Log`.

In [8]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Support execution from the project root or from notebooks/.
raw_candidates = [
    Path.cwd() / 'data' / 'raw',
    Path.cwd().parent / 'data' / 'raw',
    Path('data/raw'),
    Path('../data/raw'),
]
RAW_DIR = next((p.resolve() for p in raw_candidates if p.exists()), None)
if RAW_DIR is None:
    raise FileNotFoundError('Folder data/raw tidak ditemukan.')

DATA_FILES = sorted([
    p for p in RAW_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {'.csv', '.xlsx', '.xls'} and p.name.lower() not in {'koreksi_manual.csv', 'correction_manual.csv'}
], key=lambda p: p.name.lower())
assert len(DATA_FILES) == 9, f'Diharapkan 9 file data, ditemukan {len(DATA_FILES)}.'

def read_raw(path):
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    return pd.read_csv(path)

data = {path.name: read_raw(path) for path in DATA_FILES}
display(Markdown(f'**Folder sumber:** `{RAW_DIR}` — **{len(data)} file**'))

**Folder sumber:** `D:\7. PROYEK\5. BACKTEST & OPTIMIZE\Ori_Data\data\raw` — **9 file**

In [9]:
def temporal_range(df):
    candidates = [c for c in df.columns if re.search(r'date|tanggal|periode|quartal|quarter', str(c), flags=re.I)]
    for col in candidates:
        s = df[col].dropna().astype(str).str.strip()
        if s.empty:
            continue
        # Handle labels such as 2016 Q1 and 2016-Q1.
        q = s.str.replace('-', ' ', regex=False).str.replace(r'\s+', ' ', regex=True)
        if q.str.match(r'^\d{4} Q[1-4]$').mean() >= 0.8:
            periods = pd.PeriodIndex([pd.Period(year=int(m.group(1)), quarter=int(m.group(2)), freq='Q') for value in q if (m := re.match(r'^(\d{4}) Q([1-4])$', value))], freq='Q')
            return col, str(periods.min()), str(periods.max())
        iso_dt = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
        if iso_dt.notna().mean() >= 0.8:
            return col, iso_dt.min().date(), iso_dt.max().date()
        dt = pd.to_datetime(s, errors='coerce', dayfirst=True)
        if dt.notna().mean() >= 0.8:
            return col, dt.min().date(), dt.max().date()
        return col, s.iloc[-1], s.iloc[0]
    return None, None, None

for name, df in data.items():
    display(Markdown(f'## {name}'))
    date_col, date_min, date_max = temporal_range(df)
    display(pd.DataFrame({
        'Metric': ['Shape', 'Temporal column', 'Date/period range', 'Duplicate rows'],
        'Value': [str(df.shape), date_col or 'Tidak terdeteksi', f'{date_min} — {date_max}' if date_min is not None else 'Tidak terdeteksi', int(df.duplicated().sum())]
    }))
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'NaN count': df.isna().sum()}))
    display(df.describe(include='all').T)

## adj_close_10y.csv

,Metric,Value
0,Shape,"(2465, 16)"
1,Temporal column,Date
2,Date/period range,2016-09-22 — 2026-09-22
3,Duplicate rows,0


,dtype,NaN count
Date,object,0
BBCA.JK,float64,1
BBRI.JK,float64,0
BMRI.JK,float64,0
BBNI.JK,float64,0
TLKM.JK,float64,1
EXCL.JK,float64,1
ASII.JK,float64,1
UNTR.JK,float64,1
UNVR.JK,float64,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,2465,2465,2016-09-22,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BBCA.JK,2464.0,NaN,NaN,NaN,5908.525829,2008.122048,2268.361572,4304.138916,5711.157227,7719.075562,10021.738281
BBRI.JK,2465.0,NaN,NaN,NaN,2843.331438,896.630629,1154.322266,2067.213379,2832.474609,3494.083008,5216.774414
BMRI.JK,2465.0,NaN,NaN,NaN,3066.211405,1187.846852,1251.921997,2122.938965,2488.362793,4136.128906,6002.085938
BBNI.JK,2465.0,NaN,NaN,NaN,3093.296458,823.603565,1189.884888,2507.231689,3141.041992,3720.0,5026.202637
TLKM.JK,2464.0,NaN,NaN,NaN,2685.094917,377.584595,1774.756104,2402.870911,2608.678467,2972.501221,3681.838135
EXCL.JK,2464.0,NaN,NaN,NaN,2291.361243,417.709207,1179.733276,2008.056519,2217.394287,2551.905762,4490.0
ASII.JK,2464.0,NaN,NaN,NaN,4482.194002,717.500834,2106.73999,4079.454346,4472.455322,4896.823486,7044.312988
UNTR.JK,2464.0,NaN,NaN,NaN,17044.534432,4949.036436,6852.47998,13184.669678,16646.786133,20222.420898,31381.632812
UNVR.JK,2465.0,NaN,NaN,NaN,4376.594067,1980.784274,886.68335,2582.399658,3918.676758,6243.437988,7842.163086


## BI-7Day-RR_Sorted.xlsx

...

,count,unique,top,freq
Tanggal,121,121,31/08/2026,1
Inflase Bulanan (%),121,72,"0,28",6


## usdidr_10y.csv

,Metric,Value
0,Shape,"(2601, 7)"
1,Temporal column,Date
2,Date/period range,2016-09-22 — 2026-09-22
3,Duplicate rows,0


,dtype,NaN count
Date,object,0
Adj Close,float64,0
Close,float64,0
High,float64,0
Low,float64,0
Open,float64,0
Volume,int64,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,2601,2601,2016-09-22,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Adj Close,2601.0,NaN,NaN,NaN,14920.832508,1178.907302,12917.0,14097.799805,14618.0,15696.0,18190.300781
Close,2601.0,NaN,NaN,NaN,14920.832508,1178.907302,12917.0,14097.799805,14618.0,15696.0,18190.300781
High,2601.0,NaN,NaN,NaN,14981.545127,1185.384405,12984.0,14168.0,14702.0,15753.5,18222.0
Low,2601.0,NaN,NaN,NaN,14844.990641,1166.425778,12265.0,14025.242188,14528.0,15601.5,18044.300781
Open,2601.0,NaN,NaN,NaN,14927.21154,1176.120437,12918.0,14125.0,14624.850586,15696.0,18199.0
Volume,2601.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 1. Kalender perdagangan dan return `adj_close`

Tanggal yang ada di `adj_close_10y.csv` tetapi tidak ada di `index_lq45_10y.csv` diperlakukan sebagai tanggal libur bursa. Return dihitung sebagai `pct_change()` antarbaris pada harga adjusted close, lalu diringkas per tanggal dengan return absolut maksimum di seluruh saham.

In [10]:
adj = data['adj_close_10y.csv'].copy()
lq45 = data['index_lq45_10y.csv'].copy()
adj['Date'] = pd.to_datetime(adj['Date'])
lq45['Date'] = pd.to_datetime(lq45['Date'])

missing_dates = adj.loc[~adj['Date'].isin(lq45['Date']), 'Date'].sort_values()
asset_cols = [c for c in adj.columns if c != 'Date']
adj_returns = adj.sort_values('Date').set_index('Date')[asset_cols].pct_change(fill_method=None)
holiday_returns = adj_returns.loc[adj_returns.index.isin(missing_dates)]
holiday_missing_return_cells = int(holiday_returns.isna().sum().sum())
holiday_check = pd.DataFrame({
    'Date': holiday_returns.index,
    'max_abs_return': holiday_returns.abs().max(axis=1).to_numpy(),
    'nonzero_return_count': (holiday_returns.abs() > 1e-12).sum(axis=1).to_numpy(),
    'assets_checked': len(asset_cols),
    'missing_return_cells': holiday_returns.isna().sum(axis=1).to_numpy(),
})
all_holiday_returns_zero = np.isclose(holiday_returns.fillna(0.0).to_numpy(), 0.0, atol=1e-12).all()
display(Markdown(f'**Tanggal tidak ada di indeks:** {len(missing_dates)} (target 54)'))
display(Markdown(f'**Return cell NaN pada tanggal tambahan:** {holiday_missing_return_cells}. Nilai NaN tidak dihitung sebagai return non-zero; semua return yang tersedia bernilai nol.'))
display(holiday_check)
display(pd.DataFrame({
    'Check': ['Jumlah tanggal hilang = 54', 'Semua return holiday yang tersedia = 0'],
    'Result': [len(missing_dates) == 54, all_holiday_returns_zero]
}))
assert len(missing_dates) == 54
assert all_holiday_returns_zero

**Tanggal tidak ada di indeks:** 54 (target 54)

**Return cell NaN pada tanggal tambahan:** 3. Nilai NaN tidak dihitung sebagai return non-zero; semua return yang tersedia bernilai nol.

,Date,max_abs_return,nonzero_return_count,assets_checked,missing_return_cells
0,2017-01-02,0.0,0,15,0
1,2017-02-15,0.0,0,15,0
2,2017-04-19,0.0,0,15,0
3,2017-06-01,0.0,0,15,0
4,2017-06-23,0.0,0,15,0
5,2017-06-26,0.0,0,15,0
6,2017-06-27,0.0,0,15,0
7,2017-06-28,0.0,0,15,0
8,2017-06-29,0.0,0,15,0
9,2017-06-30,0.0,0,15,0


,Check,Result
0,Jumlah tanggal hilang = 54,True
1,Semua return holiday yang tersedia = 0,True


## 2. Baris `Volume = 0` pada `index_lq45_10y.csv`

In [11]:
zero_volume = lq45.loc[lq45['Volume'].eq(0)].sort_values('Date').copy()
display(Markdown(f'**Jumlah baris Volume = 0:** {len(zero_volume)}'))
display(zero_volume)

**Jumlah baris Volume = 0:** 24

,Date,Adj Close,Close,High,Low,Open,Volume
290,2017-12-01,992.159973,992.159973,1018.330017,992.159973,1017.099976,0
1026,2020-12-15,946.729980,946.729980,946.729980,946.729980,946.729980,0
1042,2021-01-12,996.440002,996.440002,996.440002,996.440002,996.440002,0
1047,2021-01-19,989.049988,989.049988,989.049988,989.049988,989.049988,0
1057,2021-02-02,938.710022,938.710022,938.710022,938.710022,938.710022,0
1065,2021-02-15,959.320007,959.320007,959.320007,959.320007,959.320007,0
1070,2021-02-22,951.239990,951.239990,951.239990,951.239990,951.239990,0
1071,2021-02-23,959.210022,959.210022,959.210022,959.210022,959.210022,0
1073,2021-02-25,952.539978,952.539978,952.539978,952.539978,952.539978,0
1077,2021-03-03,968.219971,968.219971,968.219971,968.219971,968.219971,0


## 3. Konsistensi pertumbuhan y-on-y PDB

Nilai PDB menggunakan format Indonesia: titik sebagai pemisah ribuan dan koma sebagai desimal. Pertumbuhan dihitung dari `PDB ADHK 2010.pct_change(4) * 100`. Toleransi konsistensi ditetapkan sebesar 0,05 poin persentase, sesuai pembulatan dua desimal pada kolom sumber. Kuartal hampir konstan ditandai bila perubahan antar-kuartal pada pertumbuhan y-on-y dilaporkan <= 0,10 poin persentase.

In [12]:
pdb_name = next(name for name in data if name.lower().startswith('data pdb'))
pdb = data[pdb_name].copy()

def parse_id_decimal(series):
    cleaned = (series.astype(str)
               .str.replace(r'[^0-9,.-]', '', regex=True)
               .str.replace('.', '', regex=False)
               .str.replace(',', '.', regex=False))
    return pd.to_numeric(cleaned, errors='coerce')

pdb['ADHK_2010_num'] = parse_id_decimal(pdb['PDB ADHK 2010 (Triliun Rp)'])
pdb['reported_yoy_pct'] = parse_id_decimal(pdb['Pertumbuhan y-on-y (%)'])
pdb['calculated_yoy_pct'] = pdb['ADHK_2010_num'].pct_change(4) * 100
pdb['difference_pp'] = pdb['reported_yoy_pct'] - pdb['calculated_yoy_pct']
pdb['within_0.05pp'] = pdb['difference_pp'].abs().le(0.05)
pdb['near_constant_yoy'] = pdb['reported_yoy_pct'].diff().abs().le(0.10)

pdb_comparison = pdb[['Periode', 'reported_yoy_pct', 'calculated_yoy_pct', 'difference_pp', 'within_0.05pp', 'near_constant_yoy']]
max_abs_diff_pp = pdb['difference_pp'].abs().max()
yoy_consistent = pdb.loc[pdb['calculated_yoy_pct'].notna(), 'within_0.05pp'].all()
display(pdb_comparison)
display(Markdown(f'**Maksimum selisih absolut:** {max_abs_diff_pp:.6f} pp; **konsisten dalam toleransi 0,05 pp:** {yoy_consistent}'))
display(Markdown('### Kuartal dengan pertumbuhan y-on-y hampir konstan'))
display(pdb.loc[pdb['near_constant_yoy'], ['Periode', 'reported_yoy_pct', 'calculated_yoy_pct', 'difference_pp']])

,Periode,reported_yoy_pct,calculated_yoy_pct,difference_pp,within_0.05pp,near_constant_yoy
0,2016 Q1,4.92,NaN,NaN,False,False
1,2016 Q2,5.18,NaN,NaN,False,False
2,2016 Q3,5.02,NaN,NaN,False,False
3,2016 Q4,4.94,NaN,NaN,False,True
4,2017 Q1,5.01,5.011283,-0.001283,True,True
5,2017 Q2,5.01,5.006550,0.003450,True,True
6,2017 Q3,5.06,5.061159,-0.001159,True,True
7,2017 Q4,5.19,5.195528,-0.005528,True,False
8,2018 Q1,5.06,5.060006,-0.000006,True,False
9,2018 Q2,5.27,5.266851,0.003149,True,False


**Maksimum selisih absolut:** 0.020480 pp; **konsisten dalam toleransi 0,05 pp:** True

### Kuartal dengan pertumbuhan y-on-y hampir konstan

,Periode,reported_yoy_pct,calculated_yoy_pct,difference_pp
3,2016 Q4,4.94,NaN,NaN
4,2017 Q1,5.01,5.011283,-0.001283
5,2017 Q2,5.01,5.006550,0.003450
6,2017 Q3,5.06,5.061159,-0.001159
10,2018 Q3,5.17,5.168531,0.001469
11,2018 Q4,5.18,5.179153,0.000847
13,2019 Q2,5.05,5.058885,-0.008885
14,2019 Q3,5.02,5.016152,0.003848
15,2019 Q4,4.97,4.974450,-0.004450
24,2022 Q1,5.02,5.012396,0.007604


## 4. Nilai ekstrem inflasi dan neraca perdagangan

Pemeriksaan memakai dua tampilan: outlier formal berdasarkan pagar IQR (1,5 x IQR) dan lima nilai tertinggi/terendah. Nilai ekstrem tidak dihapus dari data mentah.

In [13]:
def parse_id_thousands(series):
    cleaned = (series.astype(str)
               .str.replace(r'[^0-9-]', '', regex=True))
    return pd.to_numeric(cleaned, errors='coerce')

def extreme_review(df, date_col, value_col, parser, label):
    work = df[[date_col, value_col]].copy()
    work['value_numeric'] = parser(work[value_col])
    q1, q3 = work['value_numeric'].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = work.loc[(work['value_numeric'] < lower) | (work['value_numeric'] > upper)].copy()
    summary = pd.DataFrame({
        'Dataset': [label], 'Min': [work['value_numeric'].min()], 'Max': [work['value_numeric'].max()],
        'Q1': [q1], 'Q3': [q3], 'IQR lower fence': [lower], 'IQR upper fence': [upper],
        'IQR outlier count': [len(outliers)]
    })
    ranked = pd.concat([
        work.nlargest(5, 'value_numeric').assign(rank_group='Top 5'),
        work.nsmallest(5, 'value_numeric').assign(rank_group='Bottom 5'),
    ]).drop_duplicates(subset=[date_col]).sort_values('value_numeric', ascending=False)
    display(Markdown(f'### {label}'))
    display(summary)
    display(Markdown('**Outlier IQR**'))
    display(outliers if not outliers.empty else pd.DataFrame({'Result': ['Tidak ada outlier berdasarkan pagar IQR']}))
    display(Markdown('**Nilai tertinggi dan terendah**'))
    display(ranked)
    return work, summary.iloc[0], outliers, ranked

inflation_name = next(name for name in data if name.lower().startswith('tingkat inflasi'))
trade_name = next(name for name in data if name.lower().startswith('neraca perdagangan'))
inflation_work, inflation_summary, inflation_outliers, inflation_ranked = extreme_review(
    data[inflation_name], 'Tanggal', 'Inflase Bulanan (%)', parse_id_decimal, 'Inflasi bulanan'
)
trade_work, trade_summary, trade_outliers, trade_ranked = extreme_review(
    data[trade_name], 'Tanggal', 'Nilai Neraca Perdagangan', parse_id_thousands, 'Neraca perdagangan'
)

### Inflasi bulanan

,Dataset,Min,Max,Q1,Q3,IQR lower fence,IQR upper fence,IQR outlier count
0,Inflasi bulanan,-0.76,1.65,0.04,0.39,-0.485,0.915,6


**Outlier IQR**

,Tanggal,Inflase Bulanan (%),value_numeric
8,31/12/2025,"-0,76",-0.76
10,31/10/2025,"1,65",1.65
11,30/09/2025,"1,17",1.17
47,30/09/2022,"0,95",0.95
52,30/04/2022,"1,17",1.17
104,31/12/2017,"0,97",0.97


**Nilai tertinggi dan terendah**

...

,Dataset,Min,Max,Q1,Q3,IQR lower fence,IQR upper fence,IQR outlier count
0,Neraca perdagangan,-2300000,7578500,300000.0,3369850.0,-4304775.0,7974625.0,0


**Outlier IQR**

,Result
0,Tidak ada outlier berdasarkan pagar IQR


**Nilai tertinggi dan terendah**

,Tanggal,Nilai Neraca Perdagangan,value_numeric,rank_group
51,30/04/2022,"Rp7,578,500",7578500,Top 5
57,31/10/2021,"Rp5,826,000",5826000,Top 5
47,31/08/2022,"Rp5,640,400",5640400,Top 5
11,31/08/2025,"Rp5,573,300",5573300,Top 5
45,31/10/2022,"Rp5,573,200",5573200,Top 5
99,30/04/2018,"-Rp1,700,000",-1700000,Bottom 5
93,31/10/2018,"-Rp1,800,000",-1800000,Bottom 5
96,31/07/2018,"-Rp2,000,000",-2000000,Bottom 5
92,30/11/2018,"-Rp2,100,000",-2100000,Bottom 5
87,30/04/2019,"-Rp2,300,000",-2300000,Bottom 5


## Data Quality Log

Keputusan di bawah berlaku untuk analisis lanjutan. File di `data/raw` tetap dipertahankan apa adanya.

In [14]:
nan_counts = {name: int(df.isna().sum().sum()) for name, df in data.items()}
quality_log = pd.DataFrame([
    {
        'Masalah': f'NaN pada data sumber: adj_close ({nan_counts["adj_close_10y.csv"]}) dan summary_metrics ({nan_counts["summary_metrics.csv"]}).',
        'Dampak': 'Perhitungan statistik, return, atau rasio tertentu dapat memiliki observasi yang hilang.',
        'Keputusan penanganan': 'Pertahankan raw; perlakukan sebagai missing dan dokumentasikan/imputasi hanya di data/processed sesuai kebutuhan.'
    },
    {
        'Masalah': f'{len(missing_dates)} tanggal pada adj_close tidak ada di kalender indeks LQ45.',
        'Dampak': 'Kalender harga lebih panjang daripada kalender indeks dan dapat mengganggu penyelarasan return.',
        'Keputusan penanganan': 'Gunakan kalender indeks untuk analisis trading-day; tanggal tambahan dipertahankan karena return seluruh aset terverifikasi nol.'
    },
    {
        'Masalah': f'{len(zero_volume)} baris index_lq45 memiliki Volume = 0.',
        'Dampak': 'Indikator berbasis volume dapat bias atau tidak informatif pada tanggal tersebut.',
        'Keputusan penanganan': 'Tandai sebagai observasi volume nol; jangan imputasi di raw dan evaluasi pengecualian pada model volume.'
    },
    {
        'Masalah': f'Pertumbuhan PDB y-on-y vs pct_change(4) memiliki selisih maksimum {max_abs_diff_pp:.6f} pp.',
        'Dampak': 'Tidak ada perbedaan material setelah mempertimbangkan pembulatan dua desimal.',
        'Keputusan penanganan': 'Terima sebagai konsisten dalam toleransi 0,05 pp; gunakan hitungan pct_change(4) sebagai kolom validasi.'
    },
    {
        'Masalah': 'Beberapa kuartal PDB memiliki perubahan y-on-y antar-kuartal <= 0,10 pp.',
        'Dampak': 'Variasi yang rendah dapat mencerminkan pembulatan atau pertumbuhan yang relatif datar.',
        'Keputusan penanganan': 'Tandai untuk review; pertahankan nilai sumber dan jangan menghaluskan seri.'
    },
    {
        'Masalah': f'Inflasi memiliki {len(inflation_outliers)} outlier berdasarkan pagar IQR, dengan rentang {inflation_work["value_numeric"].min():.2f} sampai {inflation_work["value_numeric"].max():.2f}.',
        'Dampak': 'Rata-rata dan model sensitif outlier dapat terpengaruh oleh shock/seasonality.',
        'Keputusan penanganan': 'Pertahankan sebagai kejadian ekonomi; gunakan transformasi atau winsorization hanya bila dibutuhkan dan dokumentasikan di processed.'
    },
    {
        'Masalah': f'Neraca perdagangan tidak memiliki outlier formal berdasarkan IQR; nilai ekstrem berada pada {trade_work["value_numeric"].min():,.0f} dan {trade_work["value_numeric"].max():,.0f}.',
        'Dampak': 'Rentang dan tanda surplus/defisit yang besar dapat memengaruhi skala model.',
        'Keputusan penanganan': 'Pertahankan nilai asli setelah parsing satuan ribuan; gunakan scaling/robust method pada analisis lanjutan bila relevan.'
    },
])
display(quality_log)

,Masalah,Dampak,Keputusan penanganan
0,NaN pada data sumber: adj_close (8) dan summar...,"Perhitungan statistik, return, atau rasio tert...",Pertahankan raw; perlakukan sebagai missing da...
1,54 tanggal pada adj_close tidak ada di kalende...,Kalender harga lebih panjang daripada kalender...,Gunakan kalender indeks untuk analisis trading...
2,24 baris index_lq45 memiliki Volume = 0.,Indikator berbasis volume dapat bias atau tida...,Tandai sebagai observasi volume nol; jangan im...
3,Pertumbuhan PDB y-on-y vs pct_change(4) memili...,Tidak ada perbedaan material setelah mempertim...,"Terima sebagai konsisten dalam toleransi 0,05 ..."
4,Beberapa kuartal PDB memiliki perubahan y-on-y...,Variasi yang rendah dapat mencerminkan pembula...,Tandai untuk review; pertahankan nilai sumber ...
5,Inflasi memiliki 6 outlier berdasarkan pagar I...,Rata-rata dan model sensitif outlier dapat ter...,Pertahankan sebagai kejadian ekonomi; gunakan ...
6,Neraca perdagangan tidak memiliki outlier form...,Rentang dan tanda surplus/defisit yang besar d...,Pertahankan nilai asli setelah parsing satuan ...
